In [62]:
# Main
import numpy as np
from scipy import stats
import pandas as pd

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Split and Cross Val
from sklearn.model_selection import train_test_split, GridSearchCV

# Preprocess and Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.svm import SVC
from sklearn.cluster import KMeans

# Ensemble Models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

# Metrics
from sklearn.metrics import (mean_absolute_error, 
                             accuracy_score, 
                             root_mean_squared_error, 
                             r2_score, 
                             mean_absolute_error,
                             mean_squared_error, 
                             silhouette_score)

# Environments and serialization
import joblib

# Datasets
from sklearn.datasets import load_diabetes

In [63]:
heavy_df = pd.read_csv('data/spacex_web_scraped.csv')

In [64]:
heavy_df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Date,Time,Version Booster,Booster landing
0,1,Cape Canaveral,Dragon Spacecraft Qualification Unit,U,LEO,SpaceX,Success,4 June 2010,18:45,F9v1.0,Failure (parachute)
1,2,Cape Canaveral,SpaceX COTS Demo Flight 1,U,LEO,NASA,Success,8 December 2010,15:43,F9v1.0,Failure (parachute)
2,3,Cape Canaveral,SpaceX COTS Demo Flight 2,525 kg,LEO,NASA,Success,22 May 2012,07:44,F9 v1.0,No attempt
3,4,Cape Canaveral,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,8 October 2012,00:35,F9 v1.0,No attempt
4,5,Cape Canaveral,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,1 March 2013,15:10,F9 v1.0,No attempt


In [65]:
heavy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       521 non-null    int64 
 1   Launch site      521 non-null    object
 2   Payload          518 non-null    object
 3   Payload mass     521 non-null    object
 4   Orbit            519 non-null    object
 5   Customer         507 non-null    object
 6   Launch outcome   521 non-null    object
 7   Date             521 non-null    object
 8   Time             521 non-null    object
 9   Version Booster  521 non-null    object
 10  Booster landing  521 non-null    object
dtypes: int64(1), object(10)
memory usage: 44.9+ KB


In [66]:
heavy_df['Launch outcome'].value_counts()

Launch outcome
Success    519
Failure      2
Name: count, dtype: int64

In [68]:
vandenberg_2025 = heavy_df[
    (heavy_df["Launch site"] == "Vandenberg") &
    (heavy_df["Date"].str.contains("2025", na=False))
]

print(vandenberg_2025[["Flight No.", "Launch site", "Date", "Payload"]])
print("Count:", len(vandenberg_2025))

     Flight No. Launch site               Date         Payload
329         421  Vandenberg   January 10, 2025      Starshield
332         424  Vandenberg   January 14, 2025  Transporter-12
335         427  Vandenberg   January 21, 2025        Starlink
336         428  Vandenberg   January 24, 2025        Starlink
339         431  Vandenberg   February 1, 2025        Starlink
..          ...         ...                ...             ...
481         573  Vandenberg   December 4, 2025        Starlink
482         574  Vandenberg   December 7, 2025        Starlink
485         577  Vandenberg  December 10, 2025        Starlink
487         579  Vandenberg  December 14, 2025        Starlink
490         582  Vandenberg  December 17, 2025        Starlink

[64 rows x 4 columns]
Count: 64


In [ ]:
heavy_df['Booster landing'].value_counts()

Booster landing
Success ( OCISLY           161
Success ( JRTI             121
Success ( ASOG             117
Success ( LZ‑1              38
Success ( LZ‑4              26
No attempt                  18
No attempt                   9
Success ( LZ‑2               7
Controlled (ocean)           6
Failure ( OCISLY             5
Failure ( JRTI               4
Failure (parachute)          2
Failure (ocean)              2
Precluded  (drone ship)      1
Failure ( LZ‑1               1
Success( JRTI                1
Failure ( ASOG               1
Success ( LZ‑40              1
Name: count, dtype: int64

In [ ]:
heavy_df["site_flag"] = None

# Cape = 1
heavy_df.loc[
    heavy_df["Launch site"].str.contains("Cape Canaveral|Kennedy", case=False, na=False),
    "site_flag"
] = 1

# Vandenberg = 0
heavy_df.loc[
    heavy_df["Launch site"].str.contains("Vandenberg", case=False, na=False),
    "site_flag"
] = 0

In [ ]:
heavy_df.sample()

,flight no.,launch site,payload,payload mass,orbit,customer,launch outcome,date,time,version booster,booster landing,site_flag,datetime,year
153,154,Cape Canaveral,Starlink,"~16,250 kg",LEO,SpaceX,Success,14 May 2022,20:40,F9 B5,Success ( JRTI,1,2022-05-14 20:40:00,2022


In [ ]:
cape_count = heavy_df['site_flag'].sum()
vandy_count = (heavy_df["site_flag"] == 0).sum()
print("Cape launches:", cape_count)
print("Vandenberg launches:", vandy_count)

Cape launches: 365
Vandenberg launches: 156


In [76]:
print(yearly_launches.sort_values("year"))

   year  launches
0  2013         1
1  2016         1
2  2017         5
3  2018         6
4  2019         2
5  2020         1
6  2021         3
7  2022        13
